# Bài tập Tuần 1: Đại số tuyến tính cho AI
## Bài 1: Biểu diễn dữ liệu thành ma trận & độ tương đồng

### 1. Biến dữ liệu thành ma trận
Ở phần này, ta chuẩn bị dữ liệu văn bản gồm 8 câu chia đều làm 2 chủ đề:
- 4 câu đầu liên quan đến **Công nghệ & Trí tuệ nhân tạo (AI)**.
- 4 câu sau liên quan đến **Thể thao & Bóng đá**.

Sau đó, ta tiến hành xây dựng từ điển `vocab` chứa các từ duy nhất xuất hiện trong 8 câu và biểu diễn mỗi câu dưới dạng vector tần suất từ (Bag-of-Words) để tạo thành ma trận dữ liệu $X$.

In [7]:
import numpy as np
import string

cau = [
    "1. Given the ubiquity of handwritten documents in human transactions, Optical Character Recognition (OCR) of documents have invaluable practical worth",
    "2. Optical character recognition is a science that enables to translate various types of documents or images into analyzable, editable and searchable data",
    "3. During last decade, researchers have used artificial intelligence / machine learning tools to automatically analyze handwritten and printed documents in order to convert them into electronic format",
    "4. The objective of this review paper is to summarize research that has been conducted on character recognition of handwritten documents and to provide research directions",
    "5. In this Systematic Literature Review (SLR) we collected, synthesized and analyzed research articles on the topic of handwritten OCR (and closely related topics) which were published between year 2000 to 2018",
    "6. We followed widely used electronic databases by following pre-defined review protocol",
    "7. Articles were searched using keywords, forward reference searching and backward reference searching in order to search all the articles related to the topic",
    "8. This review article serves the purpose of presenting state of the art results and techniques on OCR and also provide research directions by highlighting research gaps"
]

vocab = sorted(w for s in cau for w in s.lower().split())

def to_vector(s):
    v = np.zeros(len(vocab))
    for w in s.lower().split():
        if w in vocab:
            v[vocab.index(w)] += 1
    return v

X = np.array([to_vector(s) for s in cau])
print("Kích thước ma trận X:", X.shape)

Kích thước ma trận X: (8, 192)


#### Giải thích cấu trúc ma trận $X$:
- **Hàng (Row)**: Mỗi hàng của ma trận $X$ đại diện cho một câu văn bản trong tập dữ liệu. Ở đây có 8 câu nên tương ứng với 8 hàng.
- **Cột (Column)**: Mỗi cột đại diện cho một từ cụ thể trong bộ từ vựng `vocab` (gồm các từ duy nhất đã được chuẩn hóa). Số cột bằng kích thước của từ vựng `len(vocab)`.
- **Giá trị phần tử $X[i, j]$**: Cho biết số lần xuất hiện của từ thứ $j$ trong câu thứ $i$.

### 2. Phép toán cơ bản

In [8]:
mean_X = X.mean(axis=0)

Xc = X - mean_X

print("Shape của ma trận gốc X:        ", X.shape)
print("Shape của vector trung bình mean_X:", mean_X.shape)
print("Shape của ma trận sau khi trừ Xc:  ", Xc.shape)

Shape của ma trận gốc X:         (8, 192)
Shape của vector trung bình mean_X: (192,)
Shape của ma trận sau khi trừ Xc:   (8, 192)


#### Minh họa quy tắc Broadcasting:
- Ma trận $X$ có kích thước là `(8, N_vocab)` (trong đó `N_vocab` là số từ duy nhất).
- Vector trung bình `mean_X` được tính theo cột nên có kích thước là `(N_vocab,)` (tương đương với một mảng 1 chiều).
- Khi thực hiện phép trừ `X - mean_X`, NumPy tự động mở rộng (broadcast) vector `mean_X` có kích thước `(N_vocab,)` thành một ma trận có kích thước `(8, N_vocab)` bằng cách sao chép vector này 8 lần cho 8 hàng. Nhờ đó, phép trừ từng phần tử tương ứng (element-wise subtraction) được thực hiện một cách tự động và tối ưu mà không cần dùng vòng lặp.

### 3. Cosine Similarity
$$\text{cosine\_similarity}(\mathbf{u}, \mathbf{v}) = \frac{\mathbf{u} \cdot \mathbf{v}}{\|\mathbf{u}\|_2 \|\mathbf{v}\|_2}$$

Hàm `cosine_similarity(X, Y=None)` tính toán trên batch (dạng ma trận).

In [9]:
def cosine_similarity(X, Y=None):
    if Y is None:
        Y = X
        
    X_norm = np.linalg.norm(X, axis=1, keepdims=True)
    Y_norm = np.linalg.norm(Y, axis=1, keepdims=True)
    
    Xn = X / X_norm
    Yn = Y / Y_norm
    
    return Xn @ Yn.T

sim_matrix = cosine_similarity(X)
print("Kích thước ma trận tương đồng:", sim_matrix.shape)
print("Ma trận tương đồng:\n", np.round(sim_matrix, 2))

Kích thước ma trận tương đồng: (8, 8)
Ma trận tương đồng:
 [[1.   0.3  0.19 0.36 0.18 0.   0.11 0.21]
 [0.3  1.   0.19 0.37 0.11 0.   0.11 0.14]
 [0.19 0.19 1.   0.23 0.16 0.11 0.22 0.06]
 [0.36 0.37 0.23 1.   0.37 0.05 0.21 0.51]
 [0.18 0.11 0.16 0.37 1.   0.1  0.33 0.36]
 [0.   0.   0.11 0.05 0.1  1.   0.   0.1 ]
 [0.11 0.11 0.22 0.21 0.33 0.   1.   0.17]
 [0.21 0.14 0.06 0.51 0.36 0.1  0.17 1.  ]]


### 4. Truy vấn
Viết hàm tìm kiếm nhận vào câu truy vấn từ người dùng, chuyển đổi thành vector và tìm `top_k` câu trong tập dữ liệu gốc có độ tương đồng lớn nhất.

In [10]:
def search(query, top_k=3):
    q_vec = to_vector(query).reshape(1, -1)
    
    sims = cosine_similarity(q_vec, X)[0]
    
    top_indices = np.argsort(sims)[::-1][:top_k]
    
    for i, idx in enumerate(top_indices):
        print(f"Top {i+1}: '{cau[idx]}' (Độ tương đồng: {sims[idx]:.4f})")
    print("\n")

search("Therefore, online system is considered more complex and advance, as it resolves overlapping problem of input data that is present in the offline system", top_k=3)

Top 1: '2. Optical character recognition is a science that enables to translate various types of documents or images into analyzable, editable and searchable data' (Độ tương đồng: 0.3956)
Top 2: '4. The objective of this review paper is to summarize research that has been conducted on character recognition of handwritten documents and to provide research directions' (Độ tương đồng: 0.3913)
Top 3: '8. This review article serves the purpose of presenting state of the art results and techniques on OCR and also provide research directions by highlighting research gaps' (Độ tương đồng: 0.3207)




### 5. Nhận xét
Dựa trên ma trận độ tương đồng `sim_matrix` và kết quả tìm kiếm:
1. **Cặp câu giống nhau nhất**:
   - Cặp câu có độ tương đồng cao nhất ngoài đường chéo chính là **Câu 4** (*"The objective of this review paper is to summarize..."*) và **Câu 8** (*"This review article serves the purpose of..."*) với độ tương đồng đạt **`0.51`**.
   - **Giải thích:** Sự tương đồng cao này đến từ việc cả hai câu đều chia sẻ các từ khóa học thuật quan trọng như *"review"*, *"provide"*, *"directions"* và đặc biệt là từ *"research"* xuất hiện lặp lại nhiều lần trong cả hai câu.
   - Các cặp câu có độ tương đồng cao tiếp theo là **Câu 2 & Câu 4** (`0.37`) và **Câu 4 & Câu 5** (`0.37`).
2. **Cặp câu khác biệt nhất**:
   - Một số cặp câu có độ tương đồng bằng **`0.0`** hoàn toàn, ví dụ như cặp **Câu 1 & Câu 6** hoặc **Câu 2 & Câu 6** hoặc **Câu 6 & Câu 7**.
   - **Giải thích:** Những cặp câu này hoàn toàn không có bất kỳ từ chung nào (giao của tập từ vựng bằng rỗng), ví dụ Câu 6 viết về quy trình tìm kiếm cơ sở dữ liệu (*"review protocol"*) còn Câu 1 và 2 tập trung viết về định nghĩa và giá trị của công nghệ OCR.
3. **Phân tích kết quả truy vấn (Search)**:
   - Khi chạy truy vấn với câu: *"Therefore, online system is considered more complex and advance, as it resolves overlapping problem of input data that is present in the offline system"*
   - Kết quả trả về top giống nhất lần lượt là **Câu 2** (`0.3956`), **Câu 4** (`0.3913`) và **Câu 8** (`0.3207`).
   - **Giải thích:** Độ tương đồng khoảng 0.3 - 0.4 này phần lớn được đóng góp bởi sự trùng lặp của các từ nối phổ biến (như *"is"*, *"and"*, *"the"*, *"of"*,...) do mô hình Bag-of-Words hiện tại chưa loại bỏ từ dừng (Stop words removal) hoặc sử dụng trọng số TF-IDF để giảm tầm ảnh hưởng của các từ thông dụng này.
4. **Sự phù hợp với trực giác**:
   - Kết quả này phản ánh đúng trực giác: các câu mô tả tổng quan về OCR và định hướng nghiên cứu tài liệu viết tay (như 4, 8) có sự liên kết chặt chẽ hơn và đạt điểm tương đồng cao hơn hẳn so với câu chỉ thuần mô tả kỹ thuật tìm kiếm cơ sở dữ liệu (Câu 6).